# 🔮 Predictions Generation Pipeline
## Generate Monthly Forecasts for All Health Zones & Diseases

In [ ]:
import os
import sys
import pickle
import json
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
import warnings

# 🔧 Fix: Add /app to Python path for Django imports
if '/app' not in sys.path:
    sys.path.insert(0, '/app')

# Set working directory to /app
os.chdir('/app')

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Django setup
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'core.settings')
import django
django.setup()

from analytics.models import HealthZone, Prediction, ClinicalData
from tensorflow.keras.models import load_model
from sklearn.preprocessing import RobustScaler

print("✅ All imports successful")


## 1. Load Models & Scalers

In [ ]:
MODELS_DIR = Path("/app/models")
METADATA_FILE = MODELS_DIR / "training_metadata.json"

# Load metadata
with open(METADATA_FILE, 'r') as f:
    metadata = json.load(f)

print(f"📋 Loaded training metadata")
print(f"   Models trained: {metadata['models_trained']}")
print(f"   Diseases: {list(metadata['diseases'].keys())}")

# Load models and scalers
models = {}
scalers = {}

for disease in metadata['diseases'].keys():
    disease_safe = disease.replace('/', '_')
    
    # Load model
    model_path = MODELS_DIR / f"model_{disease_safe}_tier*.keras"
    from glob import glob
    model_files = glob(str(model_path))
    
    if model_files:
        try:
            models[disease] = load_model(model_files[0])
            print(f"✓ Loaded model for {disease}")
        except Exception as e:
            print(f"⚠️ Error loading model for {disease}: {e}")
    
    # Load scaler
    scaler_path = MODELS_DIR / f"scaler_{disease_safe}.pkl"
    if scaler_path.exists():
        with open(scaler_path, 'rb') as f:
            scalers[disease] = pickle.load(f)
        print(f"✓ Loaded scaler for {disease}")

## 2. Load Latest Health Data

In [ ]:
# Get all health zones
health_zones = HealthZone.objects.all()
print(f"🏥 Total health zones: {health_zones.count()}")

# Get diseases
diseases = list(metadata['diseases'].keys())
print(f"🦠 Diseases to predict: {diseases}")

# Get latest data for each zone
window_size = 4  # 4-month window for prediction

def get_latest_data(zone, disease, window=4):
    """Get latest N months of data for zone+disease"""
    records = ClinicalData.objects.filter(
        health_zone=zone,
        disease_name=disease
    ).order_by('-date')[:window]
    
    if len(records) < window:
        return None
    
    cases = [r.cases for r in reversed(records)]
    return np.array(cases).reshape(-1, 1)

print("\n✅ Data loading functions ready")

## 3. Generate Predictions

In [ ]:
# Prediction setup
today = datetime.now().date()
prediction_date = today.replace(day=1) + timedelta(days=32)  # First day of next month
prediction_date = prediction_date.replace(day=1)

total_predictions = 0
predictions_by_disease = {disease: 0 for disease in diseases}

print(f"\n🔮 Generating predictions for: {prediction_date.strftime('%B %Y')}")
print(f"{'='*60}")

# Generate predictions for each zone + disease
for zone in health_zones:
    for disease in diseases:
        if disease not in models:
            continue
        
        # Get latest data
        data = get_latest_data(zone, disease, window_size)
        
        if data is None:
            continue
        
        # Normalize using disease-specific scaler
        try:
            scaler = scalers[disease]
            data_normalized = scaler.transform(data)
            
            # Get model
            model = models[disease]
            
            # Predict
            data_reshaped = data_normalized.reshape(1, window_size, 1)
            pred_normalized = model.predict(data_reshaped, verbose=0)[0][0]
            
            # Denormalize
            pred_denormalized = scaler.inverse_transform([[pred_normalized]])[0][0]
            pred_cases = max(0, float(pred_denormalized))  # Ensure non-negative
            
            # Store in database
            prediction_obj, created = Prediction.objects.update_or_create(
                health_zone=zone,
                disease_name=disease,
                prediction_date=prediction_date,
                defaults={
                    'predicted_cases': pred_cases,
                    'confidence': 0.85,  # Placeholder
                    'influence_factors': {
                        'model_tier': metadata['diseases'][disease]['tier'],
                        'last_updated': datetime.now().isoformat()
                    }
                }
            )
            
            total_predictions += 1
            predictions_by_disease[disease] += 1
            
            if created:
                print(f"✓ {zone.name[:30]:<30} | {disease:<15} | {pred_cases:>8.0f} cases")
        
        except Exception as e:
            print(f"⚠️ Error predicting for {zone.name} - {disease}: {e}")

print(f"\n{'='*60}")

## 4. Prediction Summary

In [ ]:
print(f"\n{'='*60}")
print("📊 PREDICTIONS GENERATED")
print(f"{'='*60}")
print(f"\nTotal predictions: {total_predictions}")
print(f"Prediction date: {prediction_date.strftime('%d %B %Y')}")
print(f"\nBreakdown by disease:")
for disease, count in predictions_by_disease.items():
    percentage = (count / total_predictions * 100) if total_predictions > 0 else 0
    print(f"  {disease:<15}: {count:>5} predictions ({percentage:>5.1f}%)")

# Top risk zones
from django.db.models import Sum
top_zones = Prediction.objects.filter(
    prediction_date=prediction_date
).values('health_zone__name').annotate(
    total_cases=Sum('predicted_cases')
).order_by('-total_cases')[:10]

print(f"\n🔴 Top 10 Risk Health Zones:")
for rank, zone_data in enumerate(top_zones, 1):
    print(f"  {rank:2}. {zone_data['health_zone__name']:<40} | {zone_data['total_cases']:>10.0f} cases")

print(f"\n✅ Predictions pipeline complete")